## Ingestão BigQuery para Azure Blob Bronze

Notebook simplificado para execução local no Jupyter ou no Databricks. O fluxo carrega configurações, inicializa os clientes, exporta tabelas do BigQuery em Parquet e grava os arquivos no contêiner Bronze.


### 1. Dependências

Execute a célula abaixo apenas se o ambiente ainda não possuir as bibliotecas necessárias. Após instalar no Jupyter, reinicie o kernel.


In [4]:
%pip install azure-storage-blob google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pyarrow python-dotenv


Note: you may need to restart the kernel to use updated packages.


In [5]:
databricks_utils = globals().get("dbutils")

if databricks_utils is not None:
    print("✅ Databricks detectado. Reiniciando o Python...")
    databricks_utils.library.restartPython()
else:
    print("ℹ️ Ambiente local detectado. Reinício ignorado.")

ℹ️ Ambiente local detectado. Reinício ignorado.


### 2. Imports e detecção do ambiente


In [6]:
from pathlib import Path
from typing import Dict, List, Optional
import atexit
import datetime as dt
import os
import sys
import tempfile
import google.cloud.bigquery as bigquery

from azure.core.credentials import AzureNamedKeyCredential
from azure.storage.blob import BlobServiceClient


def detectar_databricks() -> bool:
    try:
        dbutils.widgets
        return True
    except (NameError, AttributeError):
        return False


IS_DATABRICKS = detectar_databricks()
print("✅ Ambiente:", "Databricks" if IS_DATABRICKS else "local")
print("✅ Diretório atual:", Path.cwd())


✅ Ambiente: local
✅ Diretório atual: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/notebooks


### 3. Localizar o projeto e carregar o `.env` local


In [7]:
def localizar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()

    for candidato in [atual, *atual.parents]:
        if (candidato / "src").is_dir():
            return candidato

    raise FileNotFoundError(
        "Pasta 'src' não encontrada no diretório atual ou nos diretórios pais."
    )


BASE_DIR = localizar_raiz_projeto()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))


if not IS_DATABRICKS:
    from dotenv import load_dotenv

    env_path = BASE_DIR / ".env"

    if not env_path.is_file():
        raise FileNotFoundError(f"Arquivo .env não encontrado: {env_path}")

    if not load_dotenv(env_path, override=True):
        raise RuntimeError(f"Não foi possível carregar: {env_path}")

    print(f"✅ Ambiente local carregado: {env_path}")


from src.config.secrets import get_secret

print(f"✅ Raiz do projeto: {BASE_DIR}")


✅ Ambiente local carregado: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/.env
✅ Raiz do projeto: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28


### 4. Configurações e segredos


In [8]:
SECRET_SCOPE = "kvfiaptechprod"
DEFAULT_GCP_PROJECT_ID = "tough-medley-505300-k1"
DEFAULT_GCP_CREDENTIAL_FILE = "tough-medley-505300-k1-164371097431.json"

DEFAULT_TABLES = [
    "basedosdados.br_inep_avaliacao_alfabetizacao.uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf",
    "basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio",
    "basedosdados.br_inep_avaliacao_alfabetizacao.municipio",
    "basedosdados.br_inep_avaliacao_alfabetizacao.alunos",
]


def exigir_valor(valor: Optional[str], nome: str) -> str:
    valor = str(valor).strip() if valor is not None else ""

    if not valor:
        raise RuntimeError(f"{nome} não configurado.")

    return valor


AZURE_STORAGE_ACCOUNT = exigir_valor(
    get_secret("AZURE-STORAGE-ACCOUNT", "AZURE_STORAGE_ACCOUNT"),
    "AZURE_STORAGE_ACCOUNT",
)

AZURE_STORAGE_KEY = exigir_valor(
    get_secret("AZURE-STORAGE-KEY", "AZURE_STORAGE_KEY"),
    "AZURE_STORAGE_KEY",
)

BRONZE_CONTAINER = exigir_valor(
    get_secret("AZURE-CONTAINER-BRONZE", "AZURE_CONTAINER_BRONZE"),
    "AZURE_CONTAINER_BRONZE",
)

GOOGLE_CREDENTIALS_JSON = get_secret(
    "GOOGLE-APPLICATION-CREDENTIALS-JSON",
    "GOOGLE_APPLICATION_CREDENTIALS_JSON",
)

GCP_PROJECT_ID = os.getenv(
    "GCP_PROJECT_ID",
    DEFAULT_GCP_PROJECT_ID,
).strip()

print(f"✅ Storage Account: {AZURE_STORAGE_ACCOUNT}")
print(f"✅ Container Bronze: {BRONZE_CONTAINER}")
print(f"✅ Projeto GCP: {GCP_PROJECT_ID}")
print(f"✅ Storage Key presente: {bool(AZURE_STORAGE_KEY)}")


✅ Storage Account: stfiaptechchallenge2iast
✅ Container Bronze: bronze
✅ Projeto GCP: tough-medley-505300-k1
✅ Storage Key presente: True


### 5. Credencial Google e tabelas


In [9]:
TEMP_CREDENTIAL_PATH = None


def remover_credencial_temporaria() -> None:
    if TEMP_CREDENTIAL_PATH:
        caminho = Path(TEMP_CREDENTIAL_PATH)
        if caminho.is_file():
            caminho.unlink()


atexit.register(remover_credencial_temporaria)


def configurar_credencial_google() -> Path:
    global TEMP_CREDENTIAL_PATH

    if GOOGLE_CREDENTIALS_JSON:
        with tempfile.NamedTemporaryFile(
            mode="w",
            suffix=".json",
            delete=False,
            encoding="utf-8",
        ) as arquivo:
            arquivo.write(GOOGLE_CREDENTIALS_JSON)
            TEMP_CREDENTIAL_PATH = arquivo.name

        caminho = Path(TEMP_CREDENTIAL_PATH)
        origem = "Key Vault"
    else:
        caminho = BASE_DIR / "credenciais" / DEFAULT_GCP_CREDENTIAL_FILE
        origem = "arquivo local"

    if not caminho.is_file():
        raise FileNotFoundError(f"Credencial Google não encontrada: {caminho}")

    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = str(caminho)
    print(f"✅ Credencial Google carregada de: {origem}")
    return caminho


def carregar_tabelas() -> List[str]:
    if not IS_DATABRICKS:
        return DEFAULT_TABLES.copy()

    dbutils.widgets.text("TABLES", "")
    valor = dbutils.widgets.get("TABLES").strip()

    if not valor:
        return DEFAULT_TABLES.copy()

    tabelas = [item.strip() for item in valor.split(",") if item.strip()]
    return tabelas or DEFAULT_TABLES.copy()


GCP_CREDENTIAL_PATH = configurar_credencial_google()
TABLES = carregar_tabelas()

print(f"✅ Tabelas configuradas: {len(TABLES)}")
for tabela in TABLES:
    print(f"   • {tabela}")


✅ Credencial Google carregada de: arquivo local
✅ Tabelas configuradas: 6
   • basedosdados.br_inep_avaliacao_alfabetizacao.uf
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
   • basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_municipio
   • basedosdados.br_inep_avaliacao_alfabetizacao.municipio
   • basedosdados.br_inep_avaliacao_alfabetizacao.alunos


### 6. Inicializar e validar os clientes


In [10]:
def criar_cliente_bigquery():
    cliente = bigquery.Client.from_service_account_json(
        str(GCP_CREDENTIAL_PATH),
        project=GCP_PROJECT_ID,
    )
    print("✅ Cliente BigQuery inicializado.")
    return cliente


def criar_cliente_blob():
    credencial = AzureNamedKeyCredential(
        name=AZURE_STORAGE_ACCOUNT,
        key=AZURE_STORAGE_KEY,
    )

    cliente = BlobServiceClient(
        account_url=(
            f"https://{AZURE_STORAGE_ACCOUNT}.blob.core.windows.net"
        ),
        credential=credencial,
    )

    cliente.get_account_information()

    container = cliente.get_container_client(BRONZE_CONTAINER)
    container.get_container_properties()

    print("✅ Cliente Azure Blob autenticado.")
    print(f"✅ Container acessível: {BRONZE_CONTAINER}")
    return cliente


bq_client = criar_cliente_bigquery()
blob_service_client = criar_cliente_blob()


✅ Cliente BigQuery inicializado.
✅ Cliente Azure Blob autenticado.
✅ Container acessível: bronze


### 7. Função de exportação


In [11]:
def exportar_tabela(
    source_table: str,
    blob_name: str,
) -> Dict:
    if not source_table:
        raise ValueError(
            "Tabela de origem não informada."
        )

    if not blob_name:
        raise ValueError(
            "Nome do blob não informado."
        )

    query = (
        f"SELECT * "
        f"FROM `{source_table}`"
    )

    print(
        f"▶ Consultando: {source_table}"
    )

    dataframe = (
        bq_client
        .query(
            query,
            location="US",
        )
        .to_dataframe()
    )

    if dataframe.empty:
        raise ValueError(
            f"Nenhum registro retornado por "
            f"{source_table}."
        )

    # =========================
    # Colunas de auditoria
    # =========================

    dataframe["_ingested_at"] = (
        dt.datetime.now(
            dt.timezone.utc
        ).isoformat()
    )

    dataframe["_source_table"] = (
        source_table
    )

    # =========================
    # Pasta Bronze local
    # =========================

    local_bronze_path = (BASE_DIR/ "dados" / "bronze")
    local_bronze_path.mkdir(parents=True,exist_ok=True,)
    parquet_file = (local_bronze_path/ blob_name)

    # =========================
    # Gerar Parquet local
    # =========================

    dataframe.to_parquet(
        parquet_file,
        index=False,
    )

    if not parquet_file.is_file():
        raise FileNotFoundError(
            f"Parquet não criado: "
            f"{parquet_file}"
        )

    tamanho_local = (
        parquet_file.stat().st_size
    )

    print(
        f"✅ Parquet local: "
        f"{parquet_file}"
    )

    # =========================
    # Upload para Azure Bronze
    # =========================

    blob_client = (
        blob_service_client
        .get_blob_client(
            container=BRONZE_CONTAINER,
            blob=blob_name,
        )
    )

    with parquet_file.open("rb") as arquivo:
        blob_client.upload_blob(
            data=arquivo,
            overwrite=True,
        )

    propriedades = (
        blob_client.get_blob_properties()
    )

    if propriedades.size != tamanho_local:
        raise RuntimeError(
            "Tamanho divergente. "
            f"Local: {tamanho_local} bytes. "
            f"Azure: {propriedades.size} bytes."
        )

    resultado = {
        "success": True,
        "table": source_table,
        "container": BRONZE_CONTAINER,
        "blob": blob_name,
        "local_file": str(parquet_file),
        "records": len(dataframe),
        "columns": len(dataframe.columns),
        "size_bytes": propriedades.size,
    }

    print(
        f"✅ Blob enviado: {blob_name}"
    )

    print(
        f"✅ Registros: "
        f"{len(dataframe):,}"
    )

    print(
        f"✅ Colunas: "
        f"{len(dataframe.columns)}"
    )

    print(
        f"✅ Tamanho: "
        f"{propriedades.size / 1024:.2f} KB"
    )

    return resultado

### 8. Executar ingestão batch


In [12]:
def executar_ingestao(tabelas: List[str]) -> List[Dict]:
    data_execucao = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%d")
    resultados = []
    falhas = []

    print("========== INGESTÃO ==========")
    print(f"✅ Data UTC: {data_execucao}")
    print(f"✅ Total de tabelas: {len(tabelas)}")

    for indice, tabela in enumerate(tabelas, start=1):
        sufixo = tabela.rsplit(".", maxsplit=1)[-1]
        blob_name = f"{data_execucao}_{sufixo}.parquet"

        print()
        print(f"[{indice}/{len(tabelas)}] {tabela}")

        try:
            resultados.append(
                exportar_tabela(
                    source_table=tabela,
                    blob_name=blob_name,
                )
            )
        except Exception as exc:
            erro = f"{type(exc).__name__}: {exc}"
            falhas.append({"table": tabela, "error": erro})
            print(f"❌ Falha: {erro}")

    print()
    print("========== RESUMO ==========")
    print(f"✅ Sucessos: {len(resultados)}")
    print(f"❌ Falhas: {len(falhas)}")
    print(f"✅ Container: {BRONZE_CONTAINER}")

    if falhas:
        for falha in falhas:
            print(f"- {falha['table']}: {falha['error']}")

        raise RuntimeError(
            f"Processo concluído com {len(falhas)} falha(s)."
        )

    print("✅ Todas as tabelas foram exportadas com sucesso.")
    return resultados


resultados = executar_ingestao(TABLES)


========== INGESTÃO ==========
✅ Data UTC: 2026-10-18
✅ Total de tabelas: 6

[1/6] basedosdados.br_inep_avaliacao_alfabetizacao.uf
▶ Consultando: basedosdados.br_inep_avaliacao_alfabetizacao.uf
✅ Parquet local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-18_uf.parquet
✅ Blob enviado: 2026-10-18_uf.parquet
✅ Registros: 145
✅ Colunas: 17
✅ Tamanho: 17.46 KB

[2/6] basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
▶ Consultando: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_brasil
✅ Parquet local: /home/linux/projetos/postech-2iast-fase3-modelo-preditivo-g28/dados/bronze/2026-10-18_meta_alfabetizacao_brasil.parquet
✅ Blob enviado: 2026-10-18_meta_alfabetizacao_brasil.parquet
✅ Registros: 3
✅ Colunas: 13
✅ Tamanho: 9.22 KB

[3/6] basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
▶ Consultando: basedosdados.br_inep_avaliacao_alfabetizacao.meta_alfabetizacao_uf
✅ Parquet local: /home/linux/projetos

### 9. Visualizar o resumo


In [13]:
import pandas as pd

resumo = pd.DataFrame(resultados)
resumo


,success,table,container,blob,local_file,records,columns,size_bytes
0,True,basedosdados.br_inep_avaliacao_alfabetizacao.uf,bronze,2026-10-18_uf.parquet,/home/linux/projetos/postech-2iast-fase3-model...,145,17,17875
1,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-18_meta_alfabetizacao_brasil.parquet,/home/linux/projetos/postech-2iast-fase3-model...,3,13,9438
2,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-18_meta_alfabetizacao_uf.parquet,/home/linux/projetos/postech-2iast-fase3-model...,81,14,12401
3,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-18_meta_alfabetizacao_municipio.parquet,/home/linux/projetos/postech-2iast-fase3-model...,10704,15,222829
4,True,basedosdados.br_inep_avaliacao_alfabetizacao.m...,bronze,2026-10-18_municipio.parquet,/home/linux/projetos/postech-2iast-fase3-model...,23995,17,497738
5,True,basedosdados.br_inep_avaliacao_alfabetizacao.a...,bronze,2026-10-18_alunos.parquet,/home/linux/projetos/postech-2iast-fase3-model...,3867999,14,70857061


In [14]:
from pathlib import Path

import pyarrow.parquet as pq


PASTA_BRONZE = BASE_DIR / "dados" / "bronze"

arquivos_parquet = sorted(
    PASTA_BRONZE.glob("*.parquet")
)

if not arquivos_parquet:
    raise FileNotFoundError(
        f"Nenhum arquivo Parquet encontrado em: "
        f"{PASTA_BRONZE}"
    )

print(
    f"✅ Arquivos encontrados: "
    f"{len(arquivos_parquet)}"
)

for indice, arquivo in enumerate(
    arquivos_parquet,
    start=1,
):
    parquet = pq.ParquetFile(
        arquivo
    )

    total_linhas = (
        parquet.metadata.num_rows
    )

    total_colunas = (
        parquet.metadata.num_columns
    )

    lotes = parquet.iter_batches(
        batch_size=10
    )

    primeiro_lote = next(
        lotes,
        None,
    )

    print()
    print("=" * 100)
    print(
        f"[{indice}/{len(arquivos_parquet)}] "
        f"{arquivo.name}"
    )
    print(
        f"Total de linhas: "
        f"{total_linhas:,}"
    )
    print(
        f"Total de colunas: "
        f"{total_colunas}"
    )

    if primeiro_lote is None:
        print(
            "⚠️ Arquivo sem registros."
        )
        continue

    df_preview = (
        primeiro_lote
        .to_pandas()
    )

    print(
        f"Linhas exibidas: "
        f"{len(df_preview)}"
    )

    display(
        df_preview
    )

✅ Arquivos encontrados: 6

[1/6] 2026-10-18_alunos.parquet
Total de linhas: 3,867,999
Total de colunas: 14
Linhas exibidas: 10


,ano,id_municipio,id_escola,id_aluno,caderno,serie,rede,presenca,preenchimento_caderno,alfabetizado,proficiencia,peso_aluno,_ingested_at,_source_table
0,2023,1302603,60000963,13030738,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
1,2023,3303856,60024223,33069402,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
2,2023,4307203,60037612,43120847,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
3,2023,4305124,60038324,43096647,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
4,2024,1101500,60000242,11022611,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
5,2024,3104007,60022012,31185202,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
6,2024,3204906,60022392,32029511,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
7,2024,3550308,60026812,35405169,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
8,2024,4106902,60033492,41133447,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...
9,2024,4115705,60033757,41136547,1,2,3,0,0,0,NaN,NaN,2026-10-18T12:37:01.306899+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.a...



[2/6] 2026-10-18_meta_alfabetizacao_brasil.parquet
Total de linhas: 3
Total de colunas: 13
Linhas exibidas: 3


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_ingested_at,_source_table
0,2025,Pública,66.0,60.0,64.00,67.00,71.00,74.00,77.00,80.0,88.00,2026-10-18T12:36:25.104450+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80.0,87.37,2026-10-18T12:36:25.104450+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80.0,86.00,2026-10-18T12:36:25.104450+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...



[3/6] 2026-10-18_meta_alfabetizacao_municipio.parquet
Total de linhas: 10,704
Total de colunas: 15
Linhas exibidas: 10


...

,ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao,_ingested_at,_source_table
0,2024,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
1,2023,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
2,2024,SE,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80.0,92.84,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
3,2023,SE,Pública,31.30,38.3,45.9,53.6,61.2,68.3,74.6,80.0,88.34,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
4,2025,SE,Pública,50.00,38.0,46.0,54.0,61.0,68.0,75.0,80.0,87.00,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
5,2025,RN,Pública,48.00,44.0,51.0,57.0,64.0,70.0,75.0,80.0,85.00,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
6,2025,BA,Pública,55.00,43.0,50.0,57.0,64.0,70.0,75.0,80.0,87.00,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
7,2023,BA,Pública,36.80,43.4,50.2,57.1,63.6,69.8,75.2,80.0,84.53,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
8,2024,BA,Pública,35.96,43.4,50.2,57.1,63.6,69.8,75.2,80.0,90.04,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
9,2024,RN,Pública,39.29,43.8,50.6,57.3,63.8,69.9,75.3,80.0,77.72,2026-10-18T12:36:27.774760+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...



[5/6] 2026-10-18_municipio.parquet
Total de linhas: 23,995
Total de colunas: 17
Linhas exibidas: 10


,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingested_at,_source_table
0,2023,1100031,2,3,69.10,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
1,2023,1100072,2,3,58.20,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
3,2023,1101609,2,3,50.70,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
4,2023,1101807,2,3,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
5,2023,1302900,2,3,53.17,731.0744,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
6,2023,1303304,2,2,72.32,755.2250,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
7,2023,1500107,2,5,39.73,725.5913,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
8,2023,1501105,2,5,46.70,733.3467,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...
9,2023,1501725,2,3,73.21,758.3553,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:34.498687+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.m...



[6/6] 2026-10-18_uf.parquet
Total de linhas: 145
Total de colunas: 17
Linhas exibidas: 10


,ano,sigla_uf,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8,_ingested_at,_source_table
0,2023,AM,2,3,49.20,733.6637,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
1,2023,PB,2,2,55.23,744.8152,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
2,2023,PR,2,5,73.12,757.2146,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
3,2023,AP,2,3,41.87,732.7858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
4,2023,PE,2,5,58.95,747.4522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
5,2023,MA,2,3,56.39,749.8729,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
6,2023,MT,2,2,48.02,738.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
7,2023,CE,2,3,84.49,795.7293,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
8,2023,AL,2,5,43.88,729.7227,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
9,2023,CE,2,2,78.75,785.1658,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-10-18T12:36:22.041572+00:00,basedosdados.br_inep_avaliacao_alfabetizacao.uf
